In [1]:
import re
import pandas as pd
import numpy as np
import unicodedata
from IPython.display import display, Markdown, HTML

def load_data(path: str) -> pd.DataFrame:

    df = pd.read_csv(path, dtype=str)
    display(Markdown(f"Loaded data from `{path}` \n - Number of rows: **{len(df)}** \n - Number of columns: **{len(df.columns)}**"))
    return df

# Load the data from the CSV files
videos_df = load_data("../data/2026-06-16_11-25/videos.csv")
canais_df = load_data("../data/2026-06-16_11-25/canais.csv")
comentarios_df = load_data("../data/2026-06-16_11-25/comentarios.csv")


Loaded data from `../data/2026-06-16_11-25/videos.csv` 
 - Number of rows: **1451** 
 - Number of columns: **14**

Loaded data from `../data/2026-06-16_11-25/canais.csv` 
 - Number of rows: **30** 
 - Number of columns: **12**

Loaded data from `../data/2026-06-16_11-25/comentarios.csv` 
 - Number of rows: **10185** 
 - Number of columns: **15**

### Tratamento de dados da tabela de canais

Tabela canal:

<small>

| Campo                     | Tipo | Valores nulos | Descrição |
|--------------------------|------|---------------|-----------|
| input_canal              | str  | 0             | Nome ou identificador do canal de entrada |
| channel_id               | str  | 0             | ID único do canal no YouTube |
| title                    | str  | 0             | Título do canal |
| custom_url               | str  | 0             | URL personalizada do canal |
| description              | str  | 0             | Descrição do canal |
| published_at             | str  | 0             | Data de criação do canal |
| country                  | str  | 1             | País associado ao canal |
| subscriber_count         | str  | 0             | Quantidade de inscritos |
| view_count               | str  | 0             | Total de visualizações |
| video_count              | str  | 0             | Número de vídeos publicados |
| hidden_subscriber_count  | str  | 30            | Indicador se inscritos estão ocultos |
| timestamp_execucao       | str  | 0             | Data/hora da extração dos dados |

</small>

In [ ]:
import nltk
nltk.download('stopwords')
!python -m spacy download pt_core_news_sm

  Using cached pt_core_news_sm-3.8.0-py3-none-any.whl (13.0 MB)
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')


In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
import spacy

def get_stopwords():    
    nltk_words = set(stopwords.words('portuguese'))
    nlp = spacy.load("pt_core_news_sm")
    spacy_words = nlp.Defaults.stop_words
    return nltk_words.union(spacy_words)


def remove_accents(word):
    return "".join(c for c in unicodedata.normalize('NFD', word) if unicodedata.category(c) != 'Mn')

STOPWORDS_PT = {remove_accents(w).lower() for w in get_stopwords()}

def cast_types(df):
    int_cols = [
        "subscriber_count",
        "video_count",
        "view_count",
    ]

    for col in int_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

    for col in ["published_at","timestamp_execucao"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")

    return df

def clean_structure(df):
    before = len(df.columns)

    df = (
        df.sort_values("timestamp_execucao", ascending=False)
        .drop_duplicates(subset="channel_id", keep="first")
        .reset_index(drop=True)
    )
    # Remove rows with missing values in critical columns
    df = df.dropna(subset=["channel_id","input_canal","title"])

    # Strip whitespace from string columns
    df["channel_id"] = df["channel_id"].str.strip()
    df["input_canal"] = df["input_canal"].str.strip()
    df["title"] = df["title"].str.strip()
    df["custom_url"] = df["custom_url"].str.strip()

    after = len(df.columns)
    display(Markdown(f"Cleaned data structure \n - Columns before: **{before}** \n - Columns after: **{after}**"))
    return df


def normalize_text(text: str, keep_punct: bool = True, remove_stopwords: bool = False) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = text.encode("ascii", "ignore").decode("ascii")      # remove emojis
    if keep_punct:
        text = re.sub(r"[^\w\s.,!?;:()\-@#]", " ", text)
    else:
        text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = text.lower()

    if remove_stopwords:
        words = text.split()
        words = [w for w in words if w not in STOPWORDS_PT]
        text = " ".join(words)

    return text


def normalize_text_columns(df: pd.DataFrame) -> pd.DataFrame:
    text_cols = ["input_canal", "title", "custom_url"]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].str.lower().str.strip()
    return df

def validate(df : pd.DataFrame) -> pd.DataFrame:
    # Check for missing values in critical columns
    critical_cols = ["channel_id", "input_canal", "title"]
    for col in critical_cols:
        if df[col].isnull().any():
            display(Markdown(f"Warning: Missing values found in column `{col}`. Consider dropping or imputing these rows."))
    
    # Check for duplicate channel_ids
    if df["channel_id"].duplicated().any():
        display(Markdown("Warning: Duplicate `channel_id` values found. Consider dropping duplicates or investigating further."))

    return df

def export(df: pd.DataFrame, path: str, nlp_corpus_path: str = None)->None:
    df.to_csv(path, index=False) 
    display(Markdown(f"Exported cleaned data to `{path}`"))


[nltk_data] Downloading package stopwords to /home/kilmer/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [8]:
processed_canais_df = cast_types(canais_df)
processed_canais_df = clean_structure(cast_types(processed_canais_df))
processed_canais_df = normalize_text_columns(processed_canais_df)
processed_canais_df = validate(processed_canais_df)
processed_canais_df.head()["description"]

Cleaned data structure 
 - Columns before: **12** 
 - Columns after: **12**

0    Focado no mundo da tecnologia, o Canaltech é u...
1    O maior site sobre tecnologia do Brasil mantém...
2    A Full Cycle ajuda desenvolvedores a desenvolv...
3            o maior boteco de tecnologia do youtube\n
4    Maior comunidade de Flutter/Dart do Brasil\nww...
Name: description, dtype: str